In [1]:
import numpy as np
import pandas as pd
from post_processing import prepare_post_processing
from tqdm import tqdm

In [2]:
top_k_path = "experiments/babyLFM5k/output/iteration_1_top_100.tsv"
tracks_path = "experiments/babyLFM5k/input/tracks.tsv"
dataset_path = "experiments/babyLFM5k/input/dataset_filtered.inter"

In [3]:
tracks, top_k, dataset = prepare_post_processing(top_k_path, tracks_path, dataset_path)

In [4]:
l=0.25
target_distribution = "interactions"

In [5]:
if target_distribution == "interactions":
    target_distribution = dataset.groupby(["country"]).size() / dataset.shape[0]
elif target_distribution == "catalog":
    target_distribution = tracks.groupby(["country", "gender"]).size() / tracks.shape[0]
else:
    raise ValueError("Invalid target distribution. Choose 'interactions' or 'catalog'.")
    

In [6]:
# Build a group index mapping -> integer position in target array
group_index = {key: i for i, key in enumerate(target_distribution.index)}
G = len(target_distribution)
sqrt_target = np.sqrt(target_distribution.values)  # shape (G,) — computed once

In [9]:
results = []

for user in tqdm(top_k["user_id"].unique(), desc="Re-ranking users"):
    user_top_k = top_k[top_k["user_id"] == user].reset_index(drop=True)

    # Map each candidate to its group index (-1 if unknown)
    candidate_groups = np.array([
        group_index.get(row["country"], -1)
        for _, row in user_top_k.iterrows()
    ], dtype=np.int32)
    relevance = user_top_k["normalized_score"].to_numpy()

    # Cumulative discounted exposure vector over groups
    current_exposure = np.zeros(G, dtype=np.float64)
    selected_mask = np.zeros(len(user_top_k), dtype=bool)

    for position in range(1, 11):
        discount = 1.0 / np.log2(position + 1)
        candidate_mask = ~selected_mask
        if not candidate_mask.any():
            break

        # --- Vectorized Hellinger over all candidates at once ---
        # hyp_exposure[c] = current_exposure + discount * one_hot(group[c])
        # shape: (N_cands, G)
        hyp_exposure = np.tile(current_exposure, (candidate_mask.sum(), 1))
        cand_indices = np.where(candidate_mask)[0]
        cand_groups = candidate_groups[cand_indices]

        # Add discount only for candidates with a known group
        known = cand_groups >= 0
        hyp_exposure[known, cand_groups[known]] += discount

        totals = hyp_exposure.sum(axis=1, keepdims=True)
        totals[totals == 0] = 1.0  # avoid division by zero
        hyp_dist = hyp_exposure / totals  # (N_cands, G)

        # Squared Hellinger: 0.5 * sum((sqrt(target) - sqrt(hyp))^2, axis=1)
        hellinger_sq = 0.5 * np.sum((sqrt_target - np.sqrt(hyp_dist)) ** 2, axis=1)

        scores = (1 - l) * relevance[cand_indices] - l * hellinger_sq
        best_local = np.argmax(scores)
        best_idx = cand_indices[best_local]

        # Commit chosen item
        selected_mask[best_idx] = True
        g = candidate_groups[best_idx]
        if g >= 0:
            current_exposure[g] += discount

        chosen = user_top_k.iloc[best_idx].copy()
        chosen["rank"] = position
        results.append(chosen)

Re-ranking users: 100%|██████████| 882/882 [00:07<00:00, 121.33it/s]


In [8]:
results = pd.DataFrame(results)